<a href="https://colab.research.google.com/github/vidaechea/lidr-Master-ai-engineering/blob/development/session01/session_01_1_api_openai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import os
from openai import (
    OpenAI,
    AuthenticationError,
    RateLimitError,
    APIConnectionError,
    BadRequestError,
    InternalServerError,
)
from google.colab import userdata

# It's recommended to load your API key from Colab secrets.
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

client = OpenAI(api_key=OPENAI_API_KEY)

# gpt-4o-mini pricing (check periodically)
PRICING = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
}

def query_openai(message, instructions=None, model="gpt-4o-mini", temperature=0.3):
    """
    Make a call to the OpenAI Responses API and return
    the response along with relevant metadata.
    """
    try:
        response = client.responses.create(
            model=model,
            instructions=instructions,
            input=message,
            temperature=temperature,
            max_output_tokens=1000,
            store=False
        )

        # Verify the response is complete
        if response.status != "completed":
            return {"error": f"Response not completed: {response.status}"}

        # Calculate cost
        prices = PRICING.get(model, {"input": 0, "output": 0})
        cost = (
            (response.usage.input_tokens / 1_000_000) * prices["input"] +
            (response.usage.output_tokens / 1_000_000) * prices["output"]
        )

        return {
            "content": response.output_text,
            "model": response.model,
            "id": response.id,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "status": response.status,
            "cost_usd": cost,
        }

    except AuthenticationError:
        return {"error": "Invalid or missing API key"}
    except RateLimitError:
        return {"error": "Rate limit reached or insufficient credit"}
    except BadRequestError as e:
        return {"error": f"Invalid request: {e.message}"}
    except (APIConnectionError, InternalServerError):
        return {"error": "Connection or server error"}

# Usage
result = query_openai(
    message="How long would a PostgreSQL to Aurora migration take?",
    instructions="You are a software estimation consultant. Be concise."
)

if "error" in result:
    print(f"Error: {result['error']}")
else:
    print(result["content"])
    print(f"\n--- Metadata ---")
    print(f"Model: {result['model']}")
    print(f"ID: {result['id']}")
    print(f"Tokens: {result['input_tokens']} input + {result['output_tokens']} output")
    print(f"Status: {result['status']}")
    print(f"Cost: ${result['cost_usd']:.6f}")

The duration of a PostgreSQL to Aurora migration can vary based on several factors, including:

1. **Database Size**: Larger databases take longer to migrate.
2. **Complexity**: Complex schemas, stored procedures, and triggers may require more time for adaptation.
3. **Downtime Requirements**: If zero downtime is needed, the migration process will be more complex and time-consuming.
4. **Tools Used**: Using AWS Database Migration Service (DMS) can speed up the process.
5. **Testing and Validation**: Time for testing the migrated database can add to the overall duration.

**Estimation**: A small to medium-sized database might take a few days to a week, while larger or more complex migrations could take several weeks. Always plan for additional time for unforeseen issues.

--- Metadata ---
Model: gpt-4o-mini-2024-07-18
ID: resp_0b92aadd97a561ea0169e4c5a621a081a1b2811da2774cbfe2
Tokens: 32 input + 163 output
Status: completed
Cost: $0.000103
